# modular code gen: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [3_crew/community_contributions/modular_code_gen/account_directory/account_module.py](../../../../../../3_crew/community_contributions/modular_code_gen/account_directory/account_module.py)
- [3_crew/community_contributions/modular_code_gen/account_directory/accounts.py](../../../../../../3_crew/community_contributions/modular_code_gen/account_directory/accounts.py)
- [3_crew/community_contributions/modular_code_gen/account_directory/portfolio_module.py](../../../../../../3_crew/community_contributions/modular_code_gen/account_directory/portfolio_module.py)
- [3_crew/community_contributions/modular_code_gen/account_directory/share_price_module.py](../../../../../../3_crew/community_contributions/modular_code_gen/account_directory/share_price_module.py)
- [3_crew/community_contributions/modular_code_gen/account_directory/transaction_module.py](../../../../../../3_crew/community_contributions/modular_code_gen/account_directory/transaction_module.py)
- [3_crew/community_contributions/modular_code_gen/pyproject.toml](../../../../../../3_crew/community_contributions/modular_code_gen/pyproject.toml)
- [3_crew/community_contributions/modular_code_gen/README.md](../../../../../../3_crew/community_contributions/modular_code_gen/README.md)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/__init__.py](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/__init__.py)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/config/agents.yaml](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/config/agents.yaml)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/config/tasks.yaml](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/config/tasks.yaml)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/crew.py](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/crew.py)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/main.py](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/main.py)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/tools/__init__.py](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/tools/__init__.py)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/tools/custom_tool.py](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/tools/custom_tool.py)
- [3_crew/community_contributions/modular_code_gen/src/modular_code_gen/tools/file_saver.py](../../../../../../3_crew/community_contributions/modular_code_gen/src/modular_code_gen/tools/file_saver.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 2. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 3. A callback receives a message and history

A UI invokes your function and displays its return value. Earlier turns are an input, not automatic model memory. This practice calls the callback directly without starting a server.

**Predict:** What changes when history contains two old messages?

In [ ]:
def chat(message, history):
    return f"You asked: {message}. Earlier messages: {len(history)}."
first = chat("Hello", [])
second = chat("Explain more", [{"role":"user", "content":"Hello"}, {"role":"assistant", "content":first}])
print(first, second, sep="\n")
assert "Earlier messages: 2" in second

**Check:** The count becomes two. Gradio wiring is gr.ChatInterface(chat, type="messages"); launch starts the server.

**Change one thing:** Make the callback return an uppercase version of the question. Compare returning a value with printing it.

## 4. Cache and simulation are not live prices

The trading exercises use market data and cached values. A fixed fictional quote lets you observe caching. A fallback simulation must stay identifiable as simulated data.

**Predict:** How many provider-fixture calls occur for the same cached symbol?

In [ ]:
from functools import lru_cache
calls = []
@lru_cache(maxsize=8)
def practice_quote(symbol):
    calls.append(symbol)
    return {"symbol": symbol, "price": 100.0, "source": "fictional fixture"}
first = practice_quote("DEMO")
second = practice_quote("DEMO")
print(first, "Fixture calls:", len(calls))
assert len(calls) == 1 and second["source"] == "fictional fixture"

**Check:** One. Cached or random fallback prices are not verified current market values.

**Change one thing:** Call a second symbol. Explain how you would record a quote timestamp and expire old cache entries.

## 5. Simulated accounts change state

Buying shares updates cash, holdings, and a transaction record. Practice against a dictionary so you do not import account modules that create or alter databases.

**Predict:** How much cash remains after buying three fictional shares at 100 each?

In [ ]:
account = {"cash": 1000.0, "holdings": {}, "transactions": []}
def buy(symbol, quantity, price):
    cost = quantity * price
    if quantity <= 0 or price <= 0 or cost > account["cash"]:
        raise ValueError("Invalid or unaffordable order")
    account["cash"] -= cost
    account["holdings"][symbol] = account["holdings"].get(symbol, 0) + quantity
    account["transactions"].append({"symbol": symbol, "quantity": quantity, "price": price})
buy("DEMO", 3, 100)
print(account)
assert account["cash"] == 700 and account["holdings"]["DEMO"] == 3

**Check:** 700. These are fictional values and this code places no real order.

**Change one thing:** Try an unaffordable order and confirm the balance is unchanged. Explain why reset is a data-changing action.

## 6. Roles, tasks, and process

CrewAI separates who does work from what work is required. A sequential process has a fixed task order; a hierarchical process gives a manager coordination responsibility.

**Predict:** Is a role description itself an executable task?

In [ ]:
agents = {"researcher": "Collect fictional evidence", "analyst": "Interpret supplied evidence"}
tasks = [{"name": "research_task", "agent": "researcher", "expected_output": "Findings"},
         {"name": "analysis_task", "agent": "analyst", "expected_output": "Short report"}]
outputs = []
for task in tasks:
    assert task["agent"] in agents
    outputs.append({"task": task["name"], "input_from_previous": list(outputs), "result": task["expected_output"]})
print([item["task"] for item in outputs])
assert len(outputs[1]["input_from_previous"]) == 1

**Check:** No. A task assigns work to a role. This dictionary simulation does not execute CrewAI.

**Change one thing:** Add a reviewer role and task. Describe whether its input should include the analyst output.

## 7. Configuration fills named placeholders

The crew entry point supplies inputs, and YAML descriptions contain placeholders. Configuration names must match the inputs used at runtime.

**Predict:** What happens if motion is not supplied?

In [ ]:
description = "Propose an argument for: {motion}."
inputs = {"motion": "Beginners should practice offline first"}
rendered = description.format(**inputs)
print(rendered)
assert inputs["motion"] in rendered
try:
    description.format(**{})
except KeyError:
    print("Missing placeholder detected")

**Check:** Formatting raises KeyError here. Crew configuration and project environment are separate from the root project.

**Change one thing:** Add a second placeholder named audience and provide its value. Identify matching input keys in the linked source main.py.

## 8. Generated code is an output before it is execution

A coding workflow designs, writes, and reviews code. Generated files or Docker execution are side effects. First inspect and validate a harmless code string locally.

**Predict:** Does parsing Python text execute it?

In [ ]:
import ast
proposed = "def add(a, b):\n    return a + b\n"
tree = ast.parse(proposed)  # Parse syntax only; do not execute generated text.
functions = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
print(functions)
assert functions == ["add"]
checks = {"has_function": True, "has_test_plan": False}
print("Review:", checks)

**Check:** No. Syntax validity does not prove behavior or make generated code safe to execute.

**Change one thing:** Write a test plan for the intended function. Explain which separate step would run code and where the source uses Docker.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.